# Partie 1: Audite des Données 

## 1) Audit initial des données

1. Objectif

Ce notebook a pour but de réaliser un premier audit exploratoire du jeu de données utilisé dans le stage.

Les objectifs sont les suivants :

- charger proprement les données ;
- vérifier leur structure générale ;
- identifier les types de variables ;
- mesurer la qualité des données (valeurs manquantes, doublons, incohérences simples) ;
- produire un premier diagnostic utile pour les étapes de prétraitement et de modélisation.

2. Sortie attendue

À l'issue de ce notebook, on doit être capable de répondre aux questions suivantes :

- quelle est la taille du jeu de données ?
- quelles sont les variables disponibles ?
- quels sont leurs types ?
- y a-t-il des valeurs manquantes ou des doublons ?
- quelles variables semblent importantes ou problématiques ?

## 2) Imports et configuration

In [1]:
# Imports standards
from pathlib import Path
import sys

# Imports de base pour la manipulation et la visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuration d'affichage pandas pour faciliter l'audit
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

# Ajout de la racine du projet au chemin Python si nécessaire
PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Imports depuis le package src
from src.data import get_project_root, get_data_dirs, ensure_directories

# Vérification rapide des chemins du projet
print("Racine du projet :", get_project_root())
print("Répertoires de données :", get_data_dirs())

# Création des dossiers standards si besoin
ensure_directories()

Racine du projet : /Users/dialloamadou/Desktop/stage_tda/private_research/cll-tda-ml-research
Répertoires de données : {'raw': PosixPath('/Users/dialloamadou/Desktop/stage_tda/private_research/cll-tda-ml-research/data/raw'), 'interim': PosixPath('/Users/dialloamadou/Desktop/stage_tda/private_research/cll-tda-ml-research/data/interim'), 'processed': PosixPath('/Users/dialloamadou/Desktop/stage_tda/private_research/cll-tda-ml-research/data/processed')}


## 3) Chargement des données

In [ ]:
# Définition du chemin vers le fichier de travail
DATA_DIRS = get_data_dirs()

# Adapter ici le nom du fichier à ton cas
FILENAME = "mon_fichier.csv"

# On suppose ici que le fichier se trouve dans data/processed
DATA_PATH = DATA_DIRS["processed"] / FILENAME

print("Chemin du fichier :", DATA_PATH)
print("Le fichier existe :", DATA_PATH.exists())

# Chargement des données
df = pd.read_csv(DATA_PATH)

# Affichage des dimensions
print("Dimensions du jeu de données :", df.shape)


## 4) Aperçu global 

### Aperçu global du jeu de données

On commence par examiner :

- les dimensions du data_set ;
- les premières lignes ;
- les noms de colonnes ;
- les types de variables.

## 5) Code: aperçu globale

In [ ]:
# Affichage des dimensions
n_lignes, n_colonnes = df.shape
print(f"Nombre de lignes : {n_lignes}")
print(f"Nombre de colonnes : {n_colonnes}")

# Aperçu des premières lignes
display(df.head())

# Liste des colonnes
print("\nColonnes du jeu de données :")
for col in df.columns:
    print("-", col)

# Informations générales sur le DataFrame
print("\nInformations générales :")
df.info()

## 6) Types des variables/Code

Cette étape permet de distinguer rapidement :

- les variables numériques ;
- les variables catégorielles / textuelles ;
- les éventuelles variables de date ;
- les variables potentiellement mal typées.

In [ ]:
# Construction d'un tableau récapitulatif des types
resume_types = pd.DataFrame({
    "colonne": df.columns,
    "type": df.dtypes.astype(str).values,
    "nb_valeurs_non_nulles": df.notna().sum().values,
    "nb_valeurs_uniques": df.nunique(dropna=True).values
})

display(resume_types.sort_values(by=["type", "colonne"]).reset_index(drop=True))

# Séparation simple entre variables numériques et non numériques
colonnes_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
colonnes_non_numeriques = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variables numériques :", len(colonnes_numeriques))
print("Variables non numériques :", len(colonnes_non_numeriques))

## 7) Analyse des valeurs manquantes/Code

On cherche ici à répondre aux questions suivantes :

- quelles colonnes contiennent des valeurs manquantes ?
- en quelle quantité ?
- quelle est la proportion de données manquantes par variable ?

In [ ]:
# Calcul du nombre et du pourcentage de valeurs manquantes par colonne
manquants = pd.DataFrame({
    "colonne": df.columns,
    "nb_manquants": df.isna().sum().values
})

manquants["pct_manquants"] = 100 * manquants["nb_manquants"] / len(df)

# Tri décroissant pour identifier rapidement les colonnes problématiques
manquants = manquants.sort_values(by="pct_manquants", ascending=False).reset_index(drop=True)

display(manquants)

# Affichage des seules colonnes avec au moins une valeur manquante
display(manquants[manquants["nb_manquants"] > 0])

## Option visuelle

In [ ]:
# Visualisation simple des colonnes ayant des valeurs manquantes
manquants_non_nuls = manquants[manquants["nb_manquants"] > 0].copy()

if len(manquants_non_nuls) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(manquants_non_nuls["colonne"], manquants_non_nuls["pct_manquants"])
    plt.xticks(rotation=90)
    plt.ylabel("Pourcentage de valeurs manquantes")
    plt.title("Valeurs manquantes par variable")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune valeur manquante détectée.")

## 8) Doublons/Code

On vérifie ici la présence de lignes dupliquées exactes.

Selon le contexte, cette vérification pourra être affinée plus tard avec des doublons métier ou des doublons sur sous-ensembles de colonnes.

In [ ]:
# Nombre de lignes dupliquées exactes
nb_doublons = df.duplicated().sum()
pct_doublons = 100 * nb_doublons / len(df)

print(f"Nombre de doublons exacts : {nb_doublons}")
print(f"Pourcentage de doublons exacts : {pct_doublons:.2f}%")

# Affichage d'un extrait si des doublons existent
if nb_doublons > 0:
    display(df[df.duplicated()].head())

## 9) Statistiques descriptives/Code

Cette section permet d'obtenir une première photographie des variables numériques et catégorielles.

In [ ]:
# Statistiques descriptives pour les variables numériques
if len(colonnes_numeriques) > 0:
    display(df[colonnes_numeriques].describe().T)
else:
    print("Aucune variable numérique détectée.")
#=========================================

# Aperçu des variables non numériques
if len(colonnes_non_numeriques) > 0:
    resume_categories = pd.DataFrame({
        "colonne": colonnes_non_numeriques,
        "nb_modalites": [df[col].nunique(dropna=True) for col in colonnes_non_numeriques]
    }).sort_values(by="nb_modalites", ascending=False)

    display(resume_categories.reset_index(drop=True))
else:
    print("Aucune variable catégorielle ou textuelle détectée.")
#==========================================


## 10) Analyse initiale de la variable cible

Lorsque la variable cible est connue, on regarde sa distribution afin d'évaluer :
- l'équilibre des classes ;
- d'éventuels problèmes de rareté ;
- les implications pour la modélisation future.

In [ ]:
# A modifier la variable "TARGET_COL" & "FILENAME"
# Adapter ici le nom de la variable cible si elle est déjà connue
TARGET_COL = None
# Exemple :
# TARGET_COL = "target"

if TARGET_COL is not None and TARGET_COL in df.columns:
    print(f"Analyse de la cible : {TARGET_COL}")
    
    distribution_cible = df[TARGET_COL].value_counts(dropna=False)
    distribution_cible_pct = df[TARGET_COL].value_counts(dropna=False, normalize=True) * 100

    resume_cible = pd.DataFrame({
        "effectif": distribution_cible,
        "pourcentage": distribution_cible_pct.round(2)
    })

    display(resume_cible)

    plt.figure(figsize=(7, 4))
    distribution_cible.plot(kind="bar")
    plt.title(f"Distribution de la variable cible : {TARGET_COL}")
    plt.ylabel("Effectif")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Variable cible non définie ou absente du jeu de données.")

## 11) Premières visualisations/Code

On produit ici quelques visualisations simples pour repérer rapidement :
- des distributions asymétriques ;
- des valeurs extrêmes ;
- des variables très peu informatives ;
- des catégories dominantes.

In [ ]:
# On limite le nombre de colonnes affichées pour garder un notebook lisible
colonnes_numeriques_a_afficher = colonnes_numeriques[:6]

for col in colonnes_numeriques_a_afficher:
    plt.figure(figsize=(6, 4))
    df[col].dropna().hist(bins=30)
    plt.title(f"Distribution de {col}")
    plt.xlabel(col)
    plt.ylabel("Effectif")
    plt.tight_layout()
    plt.show()

## 12) Modalités principales pour variables catégorielles

In [ ]:
# Visualisation des catégories les plus fréquentes pour quelques variables non numériques
colonnes_categories_a_afficher = colonnes_non_numeriques[:4]

for col in colonnes_categories_a_afficher:
    plt.figure(figsize=(7, 4))
    df[col].astype(str).value_counts(dropna=False).head(10).plot(kind="bar")
    plt.title(f"Top 10 des modalités de {col}")
    plt.xlabel(col)
    plt.ylabel("Effectif")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 13) Synthèse d'autit (intermédiaire)

À ce stade, on résume les principaux constats de l'audit :

- qualité générale des données ;
- variables problématiques ;
- premières pistes de prétraitement ;
- points nécessitant une clarification métier ou scientifique.

## Résumé automatique simple

In [ ]:
# Construction d'un mini résumé automatique de l'audit
resume_audit = {
    "nombre_lignes": len(df),
    "nombre_colonnes": df.shape[1],
    "nombre_variables_numeriques": len(colonnes_numeriques),
    "nombre_variables_non_numeriques": len(colonnes_non_numeriques),
    "nombre_doublons_exacts": int(df.duplicated().sum()),
    "nombre_colonnes_avec_manquants": int((df.isna().sum() > 0).sum())
}

print("Résumé de l'audit :")
for cle, valeur in resume_audit.items():
    print(f"- {cle} : {valeur}")

## Conclusion

Premiers constats à compléter manuellement après exécution :

- ...
- ...
- ...

### Prochaines étapes proposées

- définir la stratégie de prétraitement ;
- corriger ou documenter les colonnes problématiques ;
- préparer une version nettoyée du jeu de données ;
- lancer une exploration plus ciblée sur les variables d'intérêt.